# 03 — Baselines completos em treino e validação

Este notebook executa um baseline por vez. O teste interno e o FUSeg permanecem bloqueados. Comece pela U-Net/ResNet-34 aleatória; somente depois de revisar seu resultado execute a DeepResUNet histórica.

In [ ]:
from pathlib import Path
import os
from google.colab import drive

drive.mount("/content/drive")
candidates = [
    Path("/content/drive/MyDrive/Projeto_TCC_Lucas"),
    Path("/content/drive/MyDrive/TCC/Projeto_TCC_Lucas"),
]
PROJECT_ROOT = next((path for path in candidates if path.is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(candidates)
os.chdir(PROJECT_ROOT)
print(f"Projeto: {PROJECT_ROOT}")

## Dependências e saída contínua

In [ ]:
import subprocess
import sys


def run_stream(command):
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    returncode = process.wait()
    print(f"\nreturncode={returncode}")
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, command)
    return returncode


run_stream([sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"])

## Cache temporário do dataset

In [ ]:
import shutil

DATA_ROOT = Path("/content/tcc_data")
shutil.copytree(PROJECT_ROOT / "Dataset", DATA_ROOT / "Dataset", dirs_exist_ok=True)
print(f'Dados temporários: {DATA_ROOT / "Dataset"}')

## Escolher o baseline

Mantenha a primeira opção na execução inicial. Para o baseline histórico, troque pela segunda linha e execute novamente a partir desta célula.

In [ ]:
CONFIG_RELATIVE = Path("configs/baselines/unet_resnet34_random.yaml")
# CONFIG_RELATIVE = Path('configs/baselines/deep_res_unet_historical.yaml')
CONFIG_PATH = PROJECT_ROOT / CONFIG_RELATIVE
print(f"Configuração selecionada: {CONFIG_PATH}")

## Testes antes de usar a GPU

In [ ]:
run_stream(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
        "tests/test_baseline_config.py",
        "tests/test_training_components.py",
        "tests/test_deep_res_unet.py",
    ]
)

## Treinar

A execução salva `last.pt` a cada época e encerra por no máximo 50 épocas ou early stopping. A seleção usa somente o Dice médio de validação em 0,5.

In [ ]:
run_stream(
    [
        sys.executable,
        "scripts/run_development_baseline.py",
        "--project-root",
        str(PROJECT_ROOT),
        "--data-root",
        str(DATA_ROOT),
        "--config",
        str(CONFIG_RELATIVE),
    ]
)

## Localizar a execução criada

In [ ]:
import yaml

config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
experiment_id = config["experiment_id"]
run_root = PROJECT_ROOT / config["output"]["root"]
runs = list(run_root.glob(f"{experiment_id}_seed*"))
if not runs:
    raise FileNotFoundError(f"Execução não encontrada para {experiment_id}")
RUN_DIR = max(runs, key=lambda path: path.stat().st_mtime)
print(f"Execução: {RUN_DIR}")

## Selecionar threshold somente na validação

O melhor checkpoint é recarregado, inferido uma vez por batch e avaliado entre 0,05 e 0,95. Nenhum threshold por imagem é permitido.

In [ ]:
run_stream(
    [
        sys.executable,
        "scripts/analyze_validation_thresholds.py",
        "--project-root",
        str(PROJECT_ROOT),
        "--data-root",
        str(DATA_ROOT),
        "--run-dir",
        str(RUN_DIR),
    ]
)

## Relatórios e curvas

In [ ]:
import csv
import json
import matplotlib.pyplot as plt

training_summary = json.loads((RUN_DIR / "training_summary.json").read_text())
threshold_summary = json.loads((RUN_DIR / "threshold_selection.json").read_text())
display(training_summary)
display(threshold_summary)

with (RUN_DIR / "history.csv").open(encoding="utf-8") as stream:
    history = list(csv.DictReader(stream))
epochs = [int(row["epoch"]) for row in history]
train_loss = [float(row["train_loss_total"]) for row in history]
val_loss = [float(row["validation_loss_total"]) for row in history]
val_dice = [float(row["validation_dice_mean"]) for row in history]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, train_loss, label="treino")
axes[0].plot(epochs, val_loss, label="validação")
axes[0].set(xlabel="Época", ylabel="Loss", title="Loss")
axes[0].legend()
axes[1].plot(epochs, val_dice)
axes[1].set(xlabel="Época", ylabel="Dice médio", title="Validação em threshold 0,5")
fig.tight_layout()
fig.savefig(RUN_DIR / "training_curves.png", dpi=160, bbox_inches="tight")
plt.show()

with (RUN_DIR / "threshold_curve.csv").open(encoding="utf-8") as stream:
    curve = list(csv.DictReader(stream))
thresholds = [float(row["threshold"]) for row in curve]
fig, ax = plt.subplots(figsize=(8, 5))
for metric in ("dice", "iou", "precision", "recall"):
    ax.plot(thresholds, [float(row[f"{metric}_mean"]) for row in curve], label=metric)
ax.axvline(
    threshold_summary["selected_threshold"], color="black", linestyle="--", label="selecionado"
)
ax.set(
    xlabel="Threshold", ylabel="Média por imagem", title="Sensibilidade ao threshold — validação"
)
ax.legend()
fig.tight_layout()
fig.savefig(RUN_DIR / "threshold_curves.png", dpi=160, bbox_inches="tight")
plt.show()
print(f"Artefatos: {RUN_DIR}")

## Retomada após interrupção

Se a sessão cair, execute as células iniciais, selecione a mesma configuração, defina `RESUME = RUN_DIR / 'checkpoints/last.pt'` e use:

```python
run_stream([sys.executable, 'scripts/run_development_baseline.py', '--project-root', str(PROJECT_ROOT), '--data-root', str(DATA_ROOT), '--config', str(CONFIG_RELATIVE), '--resume', str(RESUME)])
```

## Ordem das execuções

1. A configuração com ResNet-34 aleatória gera os primeiros artefatos de referência.
2. Após a revisão, selecione a DeepResUNet histórica e repita treino, threshold e curvas.
3. Registre os resultados das referências antes da comparação com pesos ImageNet.